# Discord Mathematics Crawl

Crawlt den **Mathematics**-Server und beschränkt sich bewusst auf **vier Kanäle**:

- `calculus`  (Analysis I)
- `linear-algebra`  (Lineare Algebra I)
- `proofs-and-logic`  (Beweise & Logik)
- `computing-software`  (mathematische Software / Programmierung)

Der Ablauf ist derselbe wie beim Machine-Learning-Crawl: Login über den in den
**Colab-Secrets** hinterlegten Token (`DISCORD_TOKEN_Backupper123`), ein
REST-Client mit **Rate-Limit-Handling** und zufälligen Pausen, dann
Server → Kanäle → Nachrichten als JSON. Anschließend Merge nach Google Drive und
eine optionale Ressourcen-/Paper-Extraktion als CSV.

> **ToS-Hinweis:** Wie das ML-Original nutzt dieses Notebook einen *User*-Token
> (Self-Bot). Das Automatisieren eines User-Accounts verstößt gegen die Discord
> Terms of Service. Nutze es nur mit deinem eigenen Account, nur für Server,
> denen du beigetreten bist, und lass die Ratelimits aktiv.


In [ ]:
# Modul 1: Setup & Auth
import os
import re
import json
import time
import random
import requests
from datetime import datetime, timedelta, timezone

# --- Der 'Auth/Session Provider' Mechanismus ---
# Token sicher aus den Colab-Secrets abrufen (identisch zum ML-Crawl).
AUTH_TOKEN = None
try:
    from google.colab import userdata
    for _key in ("DISCORD_TOKEN_Backupper123", "DISCORD_TOKEN"):
        try:
            AUTH_TOKEN = userdata.get(_key)
        except Exception:
            AUTH_TOKEN = None
        if AUTH_TOKEN:
            print(f"Token erfolgreich aus Secret '{_key}' geladen.")
            break
except Exception:
    pass

if not AUTH_TOKEN:
    AUTH_TOKEN = os.environ.get("DISCORD_TOKEN_Backupper123") or os.environ.get("DISCORD_TOKEN")

if not AUTH_TOKEN:
    print("Token nicht in Secrets gefunden. Bitte unter dem Schlüssel-Symbol (links) "
          "als 'DISCORD_TOKEN_Backupper123' hinterlegen oder unten einfügen.")
    AUTH_TOKEN = input("Token: ").strip()

HEADERS = {
    # hier bitte keinen Selfbot implementieren
    "Authorization": AUTH_TOKEN,
    "Content-Type": "application/json",
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"),
}

# --- Konfiguration ---
API = "https://discord.com/api/v9"

# Ziel-Server: nach Name gesucht (Skript prueft "enthaelt"; bei mehreren Treffern
# wird der mitgliederstaerkste gewaehlt). Optional per ID pinnen.
TARGET_SERVER_NAMES = ["Mathematics"]
TARGET_SERVER_IDS = []  # z.B. ["123456789012345678"]

# NUR diese vier Kanaele werden gecrawlt.
TARGET_CHANNEL_NAMES = ["calculus", "linear-algebra", "computing-software", "proofs-and-logic"]

DAYS_TO_LOOK_BACK = 3       # wie im ML-Original; None = komplette Historie
MAX_PER_CHANNEL = 5000      # Sicherheitslimit pro Kanal
BASE_DIR = "discord_exports"

# Zufaellige Pause nach jeder Anfrage (schont die API).
MIN_DELAY, MAX_DELAY = 1.0, 2.0
TEXT_CHANNEL_TYPES = {0, 5}  # 0 = Text, 5 = Announcement

# Retry-Politik fuer transiente Fehler (Netzwerk, HTTP 5xx, Rate Limits).
MAX_TRANSIENT_RETRIES = 4    # Netzwerk / 5xx
MAX_RATE_LIMIT_RETRIES = 8   # aufeinanderfolgende HTTP 429
MAX_BACKOFF = 30.0           # Sekunden, Obergrenze fuer exponentielles Backoff
MAX_RETRY_AFTER = 60.0       # Sekunden, Obergrenze fuer 429 retry_after
print("Setup fertig.")

In [ ]:
# Modul 2: REST-Client (mit Rate-Limit-Handling) + Helpers
def _retry_after_seconds(resp):
    """429-Wartezeit aus JSON-Body oder 'Retry-After'-Header lesen (gedeckelt)."""
    value = None
    try:
        value = resp.json().get("retry_after")
    except Exception:
        value = None
    if value is None:
        header = resp.headers.get("Retry-After")
        if header:
            try:
                value = float(header)
            except ValueError:
                value = None
    try:
        value = float(value) if value is not None else 2.0
    except (TypeError, ValueError):
        value = 2.0
    return max(0.5, min(value, MAX_RETRY_AFTER))


def discord_request(method, url):
    """Einheitliche Anfrage mit Rate-Limit- und Fehler-Handling.

    Gibt None zurueck, wenn die Anfrage endgueltig fehlschlaegt (Netzwerk/5xx nach
    Retries oder nicht behebbares 4xx). None ("fehlgeschlagen") ist bewusst von
    einer leeren JSON-Liste unterscheidbar.
    """
    transient = 0     # Netzwerk / 5xx
    rate_limited = 0  # aufeinanderfolgende 429
    while True:
        try:
            resp = requests.request(method, url, headers=HEADERS, timeout=30)
        except requests.RequestException as exc:
            transient += 1
            if transient > MAX_TRANSIENT_RETRIES:
                print(f"Netzwerkfehler bei {url} (aufgegeben nach {MAX_TRANSIENT_RETRIES} Versuchen): {exc}")
                return None
            backoff = min(2 ** transient, MAX_BACKOFF)
            print(f"Netzwerkfehler bei {url}: {exc} - erneuter Versuch in {backoff:.0f}s ...")
            time.sleep(backoff)
            continue

        # Menschlicher Delay nach JEDER Anfrage.
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

        code = resp.status_code
        if code in (200, 201):
            try:
                return resp.json()
            except ValueError:
                return None
        if code == 204:
            return True
        if code == 429:  # Rate Limited
            rate_limited += 1
            if rate_limited > MAX_RATE_LIMIT_RETRIES:
                print(f"Anhaltendes Rate Limit bei {url} (aufgegeben nach {MAX_RATE_LIMIT_RETRIES} Versuchen).")
                return None
            retry_after = _retry_after_seconds(resp)
            print(f"(!) Rate Limit erreicht. Warte {retry_after:.1f}s ...")
            time.sleep(retry_after + 0.5)
            continue
        if code == 403:
            return "FORBIDDEN"  # Zugriff verweigert (z.B. interner Kanal)
        if code == 401:
            raise SystemExit("401 Unauthorized - Token ungueltig oder abgelaufen.")
        if 500 <= code < 600:  # transienter Serverfehler -> Retry mit Backoff
            transient += 1
            if transient > MAX_TRANSIENT_RETRIES:
                print(f"Serverfehler {code} bei {url} (aufgegeben nach {MAX_TRANSIENT_RETRIES} Versuchen).")
                return None
            backoff = min(2 ** transient, MAX_BACKOFF)
            print(f"Serverfehler {code} bei {url} - erneuter Versuch in {backoff:.0f}s ...")
            time.sleep(backoff)
            continue
        print(f"Fehler {code} bei {url}: {resp.text[:200]}")
        return None


def clean_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "", name).strip()


def parse_discord_timestamp(ts):
    if not ts:
        return datetime.min.replace(tzinfo=timezone.utc)
    try:
        return datetime.fromisoformat(ts)
    except ValueError:
        return datetime.strptime(ts.split(".")[0], "%Y-%m-%dT%H:%M:%S").replace(tzinfo=timezone.utc)


def resolve_target_guilds(guilds, names, ids):
    """Passende Guilds nach Name/ID finden (groesster Treffer gewinnt)."""
    result = {}
    for sid in ids:
        g = next((g for g in guilds if g["id"] == str(sid)), None)
        if g:
            result[g["id"]] = g
        else:
            print(f"  Server-ID '{sid}' nicht gefunden.")
    for name in names:
        needle = name.lower().strip()
        cand = [g for g in guilds if g["name"].lower().strip() == needle]
        if not cand:
            cand = [g for g in guilds if needle in g["name"].lower()]
        if not cand:
            print(f"  Server '{name}' nicht gefunden - Namen/ID pruefen.")
            continue
        cand.sort(key=lambda g: g.get("approximate_member_count", 0) or 0, reverse=True)
        if len(cand) > 1:
            b = cand[0]
            print(f"  Mehrere Treffer fuer '{name}', waehle groessten: "
                  f"{b['name']} ({b.get('approximate_member_count', '?')} Mitglieder).")
        result[cand[0]["id"]] = cand[0]
    return list(result.values())


def collect_channel_messages(channel_id, cutoff, max_msgs, page=100):
    """Nachrichten eines Kanals paginieren (neueste zuerst) bis Cutoff/Limit.

    Gibt (result, complete) zurueck. complete=False, wenn die Paginierung durch
    einen API-Fehler abgebrochen wurde, damit ein Teil-Export nicht als
    vollstaendig gemeldet wird.
    """
    collected, before, complete = [], None, True
    while len(collected) < max_msgs:
        url = f"{API}/channels/{channel_id}/messages?limit={page}"
        if before:
            url += f"&before={before}"
        batch = discord_request("GET", url)
        if batch == "FORBIDDEN":
            return "FORBIDDEN", False
        if batch is None:
            complete = False  # Fehler nach Retries -> als unvollstaendig markieren
            break
        if not batch:
            break  # echtes Ende (leere Liste)
        stop = False
        for m in batch:
            if cutoff is not None and parse_discord_timestamp(m["timestamp"]) <= cutoff:
                stop = True
                break
            collected.append(m)
            if len(collected) >= max_msgs:
                stop = True
                break
        before = batch[-1]["id"]
        if stop or len(batch) < page:
            break
    return collected, complete

print("REST-Client bereit.")

In [ ]:
# Modul 3: Haupt-Crawl -> Server -> Kanaele -> Nachrichten (JSON)
print("Lade Server-Liste (deine beigetretenen Guilds) ...")
my_guilds = discord_request("GET", f"{API}/users/@me/guilds?with_counts=true")

if not my_guilds or isinstance(my_guilds, str):
    raise SystemExit("Konnte Server-Liste nicht laden. Token pruefen!")

print(f"Du bist Mitglied in {len(my_guilds)} Servern.")

cutoff = None if DAYS_TO_LOOK_BACK is None else datetime.now(timezone.utc) - timedelta(days=DAYS_TO_LOOK_BACK)
if cutoff:
    print(f"Zeitfenster: Nachrichten ab {cutoff.strftime('%Y-%m-%d %H:%M UTC')}.")
else:
    print("Zeitfenster: komplette Historie.")

wanted_names = {c.lower() for c in TARGET_CHANNEL_NAMES}
targets = resolve_target_guilds(my_guilds, TARGET_SERVER_NAMES, TARGET_SERVER_IDS)
if not targets:
    raise SystemExit("Kein Ziel-Server gefunden.")

summary = {}
incomplete = []
for guild in targets:
    gname, gid = guild["name"], guild["id"]
    print(f"\n=== {gname} (ID: {gid}) ===")
    server_dir = os.path.join(BASE_DIR, clean_filename(gname))
    os.makedirs(server_dir, exist_ok=True)

    channels = discord_request("GET", f"{API}/guilds/{gid}/channels")
    if not channels or isinstance(channels, str):
        print("  Keine Kanaele lesbar (fehlende Rechte?).")
        continue

    by_name = [c for c in channels if c.get("name", "").lower() in wanted_names]
    wanted = [c for c in by_name if c.get("type") in TEXT_CHANNEL_TYPES]
    wrong_type = [c for c in by_name if c.get("type") not in TEXT_CHANNEL_TYPES]
    missing = wanted_names - {c["name"].lower() for c in by_name}
    if missing:
        print(f"  Nicht gefunden: {', '.join(sorted(missing))}")
    if wrong_type:
        print("  Gefunden, aber kein Textkanal (uebersprungen): "
              + ", ".join(f"#{c['name']} (type={c['type']})" for c in wrong_type))
    print(f"  {len(wanted)} Ziel-Kanaele: " + (", ".join('#' + c['name'] for c in wanted) or '-'))

    seen = {}
    for ch in wanted:
        time.sleep(0.5)  # kurze Pause zwischen Kanaelen
        msgs, complete = collect_channel_messages(ch["id"], cutoff, MAX_PER_CHANNEL)
        if msgs == "FORBIDDEN":
            print(f"  #{ch['name']}: kein Zugriff (403).")
            continue
        base = clean_filename(ch["name"])
        seen[base] = seen.get(base, 0) + 1
        fname = base if seen[base] == 1 else f"{base}_{ch['id']}"
        if not complete:
            fname += ".INCOMPLETE"  # Teil-Export klar kennzeichnen
        path = os.path.join(server_dir, fname + ".json")
        with open(path, "w", encoding="utf-8") as f:
            json.dump(msgs, f, indent=4, ensure_ascii=False)
        summary[ch["name"]] = summary.get(ch["name"], 0) + len(msgs)
        if not complete:
            incomplete.append(ch["name"])
        tag = "" if complete else "  [UNVOLLSTAENDIG - durch API-Fehler abgebrochen]"
        print(f"  #{ch['name']}: {len(msgs)} Nachrichten gespeichert{tag} -> {path}")

print("\nFertig! Zusammenfassung:")
for k, v in summary.items():
    mark = "  (unvollstaendig!)" if k in incomplete else ""
    print(f"  #{k}: {v} Nachrichten{mark}")
if not summary:
    print("  (nichts gespeichert)")
if incomplete:
    print("\nWARNUNG: Diese Kanaele wurden wegen API-Fehlern nur teilweise geladen "
          "(als .INCOMPLETE.json gespeichert): " + ", ".join(sorted(set(incomplete))))

In [ ]:
# Modul 4: Merge aller JSONs + Speichern auf Google Drive
import shutil

print("Verbinde Google Drive ...")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print(f"Hinweis beim Einbinden von Drive: {e}")

DRIVE_DESTINATION = "/content/drive/MyDrive/Discord_Backup_Math"
LOCAL_SOURCE = BASE_DIR

print(f"Starte Merge aus '{LOCAL_SOURCE}' ...")
master_list, file_count = [], 0
for root, dirs, files in os.walk(LOCAL_SOURCE):
    for file in files:
        if not file.endswith(".json"):
            continue
        file_path = os.path.join(root, file)
        parts = file_path.split(os.sep)
        server_name = parts[-2]
        channel_name = parts[-1].replace(".json", "")
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            for msg in data:
                msg["source_server"] = server_name
                msg["source_channel"] = channel_name
                master_list.append(msg)
            file_count += 1
        except Exception as e:
            print(f"Fehler beim Lesen von {file}: {e}")

print(f"Merge fertig! {len(master_list)} Nachrichten aus {file_count} Dateien.")

os.makedirs(DRIVE_DESTINATION, exist_ok=True)
master_path = os.path.join(DRIVE_DESTINATION, "MATH_MERGED.json")
with open(master_path, "w", encoding="utf-8") as f:
    json.dump(master_list, f, indent=4, ensure_ascii=False)
print(f"MASTER-DATEI GESPEICHERT: {master_path}")

# Optional: Einzeldateien mitkopieren
copy_folder = input("Auch die Einzel-Ordner ins Drive kopieren? (j/n): ")
if copy_folder.lower().startswith("j"):
    full_backup = os.path.join(DRIVE_DESTINATION, "Einzeldateien_Struktur")
    if os.path.exists(full_backup):
        shutil.rmtree(full_backup)
    shutil.copytree(LOCAL_SOURCE, full_backup)
    print(f"Vollstaendiges Backup kopiert nach: {full_backup}")
print("Alles erledigt.")

In [ ]:
# Modul 5: Ressourcen-/Paper-Extraktion -> CSV
import csv
try:
    import pandas as pd
except Exception:
    pd = None

INPUT_FILE = "/content/drive/MyDrive/Discord_Backup_Math/MATH_MERGED.json"
OUTPUT_CSV = "/content/drive/MyDrive/Discord_Backup_Math/math_resources.csv"

# 1) Harte Mathe-/Wissenschafts-Domains (eindeutig)
SCIENCE_DOMAINS = [
    "arxiv.org", "openreview.net", "projecteuclid.org", "ams.org",
    "mathoverflow.net", "math.stackexchange.com", "oeis.org",
    "ncatlab.org", "numdam.org", "springer.com", "sciencedirect.com",
    "tandfonline.com", "jstor.org", "doi.org", "nature.com", "ieee.org",
    "wikipedia.org/wiki", "encyclopediaofmath.org",
]
# 2) Soziale/Code-Domains (nur mit Kontext-Signal)
SOCIAL_DOMAINS = ["twitter.com", "x.com", "youtube.com", "youtu.be", "reddit.com", "github.com"]
# 3) Signalwoerter
SIGNAL_KEYWORDS = [
    "theorem", "proof", "lemma", "paper", "preprint", "abstract", "textbook",
    "lecture", "notes", "course", "problem set", "exercise", "solution",
    "calculus", "linear algebra", "eigen", "matrix", "logic", "definition",
    "integral", "derivative", "convergence", "dataset", "algorithm",
]
URL_PATTERN = re.compile(r"https?://\S+")


def contains_keyword(text):
    if not text:
        return False
    text = text.lower()
    return any(w in text for w in SIGNAL_KEYWORDS)


def extract_resources(json_path):
    leads = []
    if not os.path.exists(json_path):
        print(f"Datei nicht gefunden: {json_path}")
        return leads
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"Analysiere {len(data)} Nachrichten ...")
    for msg in data:
        content = msg.get("content", "") or ""
        embeds = msg.get("embeds", []) or []
        server = msg.get("source_server", "Unknown")
        channel = msg.get("source_channel", "Unknown")
        author = (msg.get("author") or {}).get("username", "Unknown")
        timestamp = msg.get("timestamp", "") or ""

        # A) Embeds (beste Titel-Quelle)
        for embed in embeds:
            url = embed.get("url", "") or ""
            title = embed.get("title", "") or ""
            desc = embed.get("description", "") or ""
            if not url:
                continue
            is_science = any(d in url for d in SCIENCE_DOMAINS)
            is_social_hit = any(d in url for d in SOCIAL_DOMAINS) and (
                contains_keyword(title) or contains_keyword(desc) or contains_keyword(content)
            )
            if is_science or is_social_hit:
                leads.append({
                    "Confidence": "High" if is_science else "Medium",
                    "Type": "Embed",
                    "Title": title or "No Title in Embed",
                    "Link": url,
                    "Context": (desc[:200] + "...") if desc else content[:200],
                    "Author": author,
                    "Server": server,
                    "Channel": channel,
                    "Date": timestamp[:10],
                })

        # B) Reiner Text (falls kein Embed)
        for url in URL_PATTERN.findall(content):
            if any(d in url for d in SCIENCE_DOMAINS):
                leads.append({
                    "Confidence": "High", "Type": "Direct Link",
                    "Title": "Unknown (Raw Link)", "Link": url,
                    "Context": content[:300], "Author": author,
                    "Server": server, "Channel": channel, "Date": timestamp[:10],
                })
            elif any(d in url for d in SOCIAL_DOMAINS) and contains_keyword(content):
                leads.append({
                    "Confidence": "Low/Medium", "Type": "Social Signal",
                    "Title": "Social Discussion", "Link": url,
                    "Context": content[:300], "Author": author,
                    "Server": server, "Channel": channel, "Date": timestamp[:10],
                })
    return leads


leads = extract_resources(INPUT_FILE)
if leads:
    # Duplikate (gleicher Link) entfernen
    seen_links, unique = set(), []
    for row in leads:
        if row["Link"] in seen_links:
            continue
        seen_links.add(row["Link"])
        unique.append(row)

    if pd is not None:
        df = pd.DataFrame(unique)
        df.to_csv(OUTPUT_CSV, index=False, sep=";", encoding="utf-8-sig")
        print("\n--- Vorschau (Top 5) ---")
        print(df[["Confidence", "Title", "Channel"]].head(5))
    else:
        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.DictWriter(f, fieldnames=list(unique[0].keys()), delimiter=";")
            w.writeheader()
            w.writerows(unique)
    print(f"\n{len(unique)} Ressourcen identifiziert. Gespeichert: {OUTPUT_CSV}")
else:
    print("Keine Ressourcen gefunden oder Master-Datei leer.")

In [ ]:
# Optional: CSV herunterladen
try:
    from google.colab import files
    print(f"Bereite Download vor: {OUTPUT_CSV}")
    files.download(OUTPUT_CSV)
except Exception as e:
    print(f"Download nur in Colab verfuegbar: {e}")